# AEVUM Stage 1 -- Colab training

Epoch-based training (tqdm progress bar, one logged summary line per epoch). Data and checkpoints are written directly to Google Drive as training runs, so a disconnected/idle Colab session only loses progress since the last epoch, not the whole run -- no manual archive/download step needed.

If the runtime disconnects mid-run: just re-run all cells, then add `--resume /content/drive/MyDrive/aevum/outputs/stage1_final.pt` (or a specific `stage1_epochN.pt`) to the training command in the last cell before running it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/karl4th/aevum.git /content/aevum 2>/dev/null || (cd /content/aevum && git pull)

In [ ]:
%cd /content/aevum
!pip install -q uv
!uv sync

In [ ]:
!mkdir -p /content/drive/MyDrive/aevum/data /content/drive/MyDrive/aevum/outputs

## Train

Run only after the dataset-prep cell above finished (`dataset ready: N segments`). `--epochs 100` is 100 real full passes over train-clean-100. `--resume` continues from the previous (pre-fix) run's final checkpoint on Drive rather than throwing away that progress -- remove the `--resume` line for a clean from-scratch run instead.

In [ ]:
!uv run python -u -c "
from aevum.data.librispeech import LibriSpeechSegments
ds = LibriSpeechSegments(root='/content/drive/MyDrive/aevum/data', url='train-clean-100', segment_seconds=2.0)
print(f'dataset ready: {len(ds)} segments')
"

`--librispeech-url train-clean-100` (100h, ~6.3GB download -- first run will take a while to fetch, cached on Drive after that since `--data-root` points there).

The dataset was fixed to actually cover the whole split per epoch (it used to draw one random crop per file regardless of length, so an "epoch" only touched ~16% of the audio -- see `docs/reports/stage1_v0.md`). First run against this `--data-root` will also pause for a while *before* training starts while it builds a segment index (one fast header read per file, ~28.5k files for train-clean-100) -- this is cached to `<data-root>/segment_index_*.json` afterward, so it's a one-time cost.

`--epochs 100` now means 100 real full passes over train-clean-100. `--resume` continues from the previous (pre-fix) run's final checkpoint on Drive rather than throwing away that progress -- remove the `--resume` line for a clean from-scratch run instead.

In [ ]:
!uv run python -u scripts/train_stage1.py \
  --data-root /content/drive/MyDrive/aevum/data \
  --checkpoint-dir /content/drive/MyDrive/aevum/outputs \
  --librispeech-url train-clean-100 \
  --batch-size 256 \
  --resume /content/drive/MyDrive/aevum/outputs/stage1_final.pt \
  --lr 3e-4 \
  --epochs 100